# Analyzing Network Topology

## Initialization: loading the graph

In [ ]:
import json
import networkx as nx

# JSON file path
json_file = "project/file/graph_networkx_15_nodes.json"

# Loading the topology
with open(json_file, "r") as f:
    data = json.load(f)

# Creating the graph in NetworkX
G = nx.Graph()

# Adding nodes with IDs and attributes
for node in data["nodes"]:
    node_id = node["id"]        
    attrs = {k: v for k, v in node.items() if k != "id"} 
    G.add_node(node_id, **attrs)

# Adding edges with attributes
for edge in data["links"]:
    src = edge["source"]
    dst = edge["target"]
    attrs = {k: v for k, v in edge.items() if k not in ["source", "target"]}
    G.add_edge(src, dst, **attrs)

print("Graph loaded correctly!")
print(f"Nodes: {G.nodes(data=True)}")
print(f"Edges: {G.edges(data=True)}")

: 

## Graph Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
pos = nx.spring_layout(G, seed=42)
nx.draw(
    G, pos,
    with_labels=True,
    node_size=800,
    node_color="skyblue",
    edge_color="gray",
    font_size=10
)
plt.title("Graph topology")
plt.show()

## Computing Shortest Paths

In [ ]:
all_shortest_paths = dict(nx.all_pairs_shortest_path(G))

In [ ]:
import numpy as np


def n_path_pernode(G,all_shortest_paths):
    cent= [0] * G.number_of_nodes()
    for _,i in all_shortest_paths.items():
        for _,p in i.items():
            p=p[1:-1]
            for i in p:
                cent[i]+=1
    return cent
def n_path_pernode_w_end(G,all_shortest_paths):
    cent= [0] * G.number_of_nodes()
    for _,i in all_shortest_paths.items():
        for _,p in i.items():
            p=p[1:]
            for i in p:
                cent[i]+=1
    return cent
def n_path_peredge(G,all_shortest_paths):
    cent= np.zeros((G.number_of_nodes(),G.number_of_nodes()))
    for _,i in all_shortest_paths.items():
        for _,p in i.items():
            for k in range(1,len(p)):
                cent[p[k-1]][p[k]]+=1

    return cent

def edge_visual(c):
    dict={}
    for i,row in enumerate(c):
        for j,item in enumerate(row):
            if i<j and item>0:
                dict[str(j)+","+str(i)]=c[i][j]+c[j][i]
    return dict
def analisis(G,path,title):
    pl,ax=plt.subplots(1,3,figsize=(20,7))
    pl.suptitle(title)
    ax[0].bar(np.arange(0,G.number_of_nodes()),n_path_pernode(G,path))
    ax[0].set_title("n paths per node")
    n=n_path_pernode(G,path)
    cent=n_path_peredge(G,path)
    ax[1].imshow(cent)

    hh=edge_visual(cent)
    ax[2].bar(hh.keys(),hh.values())
    ax[2].set_title("n paths per edge")
    ax[2].tick_params(axis='x',labelrotation=90)
    return n,cent

In [ ]:
n_base,cent_base=analisis(G,all_shortest_paths,"shortest path")

## Printing Shortest Paths for a specific node

In [ ]:
source_node = 0

if source_node in all_shortest_paths:
    print(f"Shortest path from node {source_node}:")
    for target, path in all_shortest_paths[source_node].items():
        print(f"  -> {target}: {path}")
else:
    print(f"{source_node} node does not exist in the graph!")

## Computing Centrality

In [ ]:
centrality = nx.betweenness_centrality(G)

print("Centrality values:")
for node, value in centrality.items():
    print(f"{node}: {value:.4f}")

## Ordering Nodes for Centrality Value (descending order)

In [ ]:
sorted_nodes = sorted(centrality.items(), key=lambda x: x[1], reverse=True)

print("Nodes ordered for descending centrality:")
for node, value in sorted_nodes:
    print(f"{node}: {value:.4f}")

## Highlighting Most Central Nodes

In [ ]:
# Central nodes are red coloured
central_nodes = set([node for node, _ in sorted_nodes[:3]])
node_colors = ["red" if n in central_nodes else "skyblue" for n in G.nodes()]

plt.figure(figsize=(8,6))
nx.draw(
    G, pos,
    with_labels=True,
    node_size=800,
    node_color=node_colors,
    edge_color="gray",
    font_size=10
)
plt.title("Graph topology with central nodes highlighted")
plt.show()

## Shortest Paths avoiding nodes with higher centrality values

In [ ]:
# Number of nodes with high centrality value to avoid
top_k = 1
nodes_to_avoid = [node for node, _ in sorted_nodes[:top_k]]

print(f"Nodes to avoid: {nodes_to_avoid}")

# New graph without these nodes
G_reduced = G.copy()
G_reduced.remove_nodes_from(nodes_to_avoid)

# Compute shortest paths again
reduced_paths = dict(nx.all_pairs_shortest_path(G_reduced))

if source_node in reduced_paths:
    print(f"New shortest paths from node {source_node} avoiding most central nodes:")
    for target, path in reduced_paths[source_node].items():
        print(f"  -> {target}: {path}")
else:
    print(f"{source_node} node does not exist in the graph!")

## Centrality values as node weights

In [ ]:
import math

# New weighted graph
G_weighted = nx.Graph()

for u, v in G.edges():
    # weight = average centrality of the two nodes + 1 to avoid 0 values
    weight = (centrality[u] + centrality[v]) / 2 
    G_weighted.add_edge(u, v, weight=weight)

# Shortest paths computation on weighted graph on single source node
weighted_paths = nx.single_source_dijkstra_path(G_weighted, source_node, weight="weight")

print(f"Shortest paths from node {source_node} minimizing centrality values:")
for target, path in weighted_paths.items():
    print(f"  -> {target}: {path}")

# Shortest paths computation on weighted graph for all nodes
all_weighted_paths = dict(nx.all_pairs_dijkstra_path(G_weighted, weight="weight"))
print(all_weighted_paths)
# print("Shortest paths for all node pairs:")
# for source, targets in all_weighted_paths.items():
#     for target, path in targets.items():
#         print(f"{source} -> {target}: {path}")

In [ ]:

n_nc,cent_nodec=analisis(G,all_weighted_paths,"path weighted on the mean centrality of the nodes")

In [ ]:
plt.imshow(cent-cent_base)

In [ ]:
ebc= nx.edge_betweenness_centrality(G)
# New weighted graph
G_weighted = nx.Graph()

for u, v in G.edges():
    # weight = average centrality of the two nodes + 1 to avoid 0 values
    weight = ebc[(u,v)]
    print(u,v,weight)
    G_weighted.add_edge(u, v, weight=weight)

all_weighted_paths_edges = dict(nx.all_pairs_dijkstra_path(G_weighted, weight="weight"))


In [ ]:
n_ec,e_ec=analisis(G,all_weighted_paths_edges,"path weighted on the centrality of the edges")
analisis(G,all_shortest_paths,"shortest distance")
;

In [ ]:
import math

# New weighted graph
G_weighted_gready= nx.Graph()
g_weight_m=np.zeros((G.number_of_nodes(),G.number_of_nodes()))
gready_paths={}
for i in range(G.number_of_nodes()):
    
    for u, v in G.edges():
        # weight = average centrality of the two nodes + 1 to avoid 0 values
        G_weighted.add_edge(u, v, weight=g_weight_m[u][v]**2)   ##!! entrambe le direzione  -> somma

    # Shortest paths computation on weighted graph on single source node
    weighted_paths = nx.single_source_dijkstra_path(G_weighted, i, weight="weight")
    gready_paths[str(i)]=weighted_paths
    g_weight_m=n_path_peredge(G,{"k":weighted_paths})
analisis(G,gready_paths,"gready approach")
analisis(G,all_shortest_paths,"base")
;



In [ ]:

g_weight_m=np.zeros((G.number_of_nodes(),G.number_of_nodes()))
gready_paths={}
for source in range(G.number_of_nodes()):
    gready_paths[str(source)]={}
    for target in range(G.number_of_nodes()):
        G_weighted_gready= nx.Graph()
        for u, v in G.edges():
            G_weighted_gready.add_edge(u, v, weight=g_weight_m[u][v])   ##!! entrambe le direzione  -> somma

        # Shortest paths computation on weighted graph on single source node
        weighted_paths = nx.dijkstra_path(G_weighted_gready, source,target, weight="weight")
        gready_paths[str(source)][str(target)]=weighted_paths
        for k in range(1,len(weighted_paths)):
            g_weight_m[weighted_paths[k-1]][weighted_paths[k]]+=1
analisis(G,gready_paths,"gready approach")

analisis(G,all_shortest_paths,"base")
;


In [ ]:
plt.imshow(cent-cent_base)

In [ ]:
fig,ax=plt.subplots()
n_show= 7
base= np.arange(0,G.number_of_nodes())[:n_show]
ax.bar(base-0.2,n_base[:n_show],width=0.2)
ax.bar(base,n_nc[:n_show],width=0.2)
ax.bar(base+0.2,n_ec[:n_show],width=0.2)

In [ ]:
+ def gready_approach(G: DiGraph, xx, yy) -> list | dict:
+     gready_paths={}
+     for source in G.nodes:
+         gready_paths[source]={}
+         for target in G.nodes:
+             weighted_paths = nx.dijkstra_path(G, source,target, weight="weight")
+             gready_paths[source][target]=weighted_paths
+             for k in range(1,len(weighted_paths)):
+                             G[weighted_paths[k-1]][weighted_paths[k]]['weight']=G[weighted_paths[k-1]][weighted_paths[k]]['weight']+1
+     #print(gready_paths)
+ 
+     return gready_paths